# 03 · THANOS DimRed: NMF vs autoencoder synergies, and a latent dynamics model

First attempt at the synergy analysis and a dynamics model (see `PROJECT_OUTLINE.md`), **revised after review**.
What changed from the original version:

| # | Problem | Fix |
|---|---|---|
| 1 | AE decoder was evaluated on the NMF latents | Separate names `H_*_nmf` / `H_*_ae`; each decoder gets its own latents |
| 2 | Failed pose frames (all 20 angles = 0) were kept | A validity mask removes them from every pose fit and score |
| 3 | 80/20 split over sorted files = 3 unseen stages by accident | Blocked split **inside each recording**: first 80% train, 1 s gap, last 20% test |
| 4 | No reference numbers | Static (median pose) baseline and a full 16-channel ridge ceiling; persistence baseline for the LSTM |
| 5 | Sequence windows crossed recording boundaries | Windows are built per recording and per block |
| 6 | The Experiment 2 pipeline was copied five times | One copy |
| 7 | Latents were unscaled; dead units unchecked | Latents standardised before the LSTM; active latent units counted |

Still **not** done here (on purpose, they belong to the experiment folders): sweeping latent size k, a
PCA / linear-AE control, multiple seeds (`notebooks/02_how_many_synergies`), and free rollouts (`notebooks/06_synergy_dynamics`).

In [ ]:
# Hardware check (works on Colab and locally)
import shutil, subprocess
if shutil.which("nvidia-smi"):
    print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)
else:
    import torch
    print("No NVIDIA GPU found" + (" (Apple MPS available)" if torch.backends.mps.is_available() else ""))

# Dataloader

In [ ]:
# Environment setup: works on Colab and locally.
# After this cell, the working directory is the emg2pose repo root.
import os, sys, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules or "COLAB_RELEASE_TAG" in os.environ or Path("/content").exists()

if IN_COLAB:
    ROOT = Path("/content/emg2pose")
    if not ROOT.exists():
        !git clone https://github.com/Wilbertbh-Tan/emg2pose.git {ROOT}
    !pip install -q h5py
else:
    # Walk up from the notebook's folder to the repo root (the folder that contains setup.py).
    ROOT = Path.cwd().resolve()
    while not (ROOT / "setup.py").exists() and ROOT != ROOT.parent:
        ROOT = ROOT.parent
    assert (ROOT / "setup.py").exists(), "Run this notebook from inside the emg2pose repo"

os.chdir(ROOT)
DATA_DIR = ROOT / "emg2pose_dataset_mini"
print(f"{'Colab' if IN_COLAB else 'Local'} | repo root: {ROOT}")

# Download the mini split (~617 MB) once; it is gitignored locally.
if not DATA_DIR.exists():
    url = "https://fb-ctrl-oss.s3.amazonaws.com/emg2pose/emg2pose_dataset_mini.tar"
    subprocess.run(["curl", "-s", url, "-o", "emg2pose_dataset_mini.tar"], check=True)
    subprocess.run(["tar", "-xf", "emg2pose_dataset_mini.tar"], check=True)
    os.remove("emg2pose_dataset_mini.tar")
print("Environment and mini-dataset ready!")

In [ ]:
import glob
import h5py
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal, stats
from scipy.stats import pearsonr
from sklearn.decomposition import NMF
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score
from sklearn.preprocessing import StandardScaler
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Running on device: {DEVICE}")

In [ ]:
# HDF5 loader + channel-quality check + envelope DSP pipeline
h5_files = sorted(glob.glob(str(DATA_DIR / "*.hdf5")))
print(f"Found {len(h5_files)} HDF5 recordings in {DATA_DIR}.")

def load_emg2pose_hdf5(filepath, max_samples=None):
    # Loads time, 16-ch sEMG (2 kHz), 20-D joint angles (radians -> degrees) and the stage label.
    with h5py.File(filepath, "r") as f:
        ts = f["emg2pose"]["timeseries"]
        stage = str(f["emg2pose"].attrs["stage"])
        sl = slice(0, max_samples) if max_samples is not None else slice(None)
        time_s = ts["time"][sl].astype(np.float64)
        emg_raw = ts["emg"][sl].astype(np.float32)               # (T, 16)
        pose_rad = ts["joint_angles"][sl].astype(np.float32)     # (T, 20)
    pose_deg = np.rad2deg(pose_rad)
    fs = 1.0 / np.median(np.diff(time_s[:1000]))
    return time_s, emg_raw, pose_deg, fs, stage

def check_channel_artifacts(emg_raw, z_thresh=4.0):
    # Per-channel log-variance and kurtosis to flag flatlined, detached, or motion-popped electrodes.
    ch_std = np.std(emg_raw, axis=0) + 1e-8
    ch_kurt = stats.kurtosis(emg_raw, axis=0)
    z_std = np.abs(stats.zscore(np.log(ch_std)))
    bad_channels = np.where((z_std > z_thresh) | (ch_std < 1e-6))[0]
    return {"ch_std": ch_std, "ch_kurtosis": ch_kurt, "flagged_channels": bad_channels.tolist()}

def preprocess_emg_envelope(emg_raw, pose_deg, fs_in=2000.0, fs_out=50.0,
                            bp_low=20.0, bp_high=450.0, notch_freq=60.0, env_lp=10.0):
    # 1. Bandpass 20-450 Hz  2. 60 Hz notch  3. Full-wave rectification
    # 4. 10 Hz low-pass envelope + decimation to 50 Hz
    # Returns ALL frames plus a validity mask for the pose (see the note below).
    nyq = 0.5 * fs_in
    high_cut = min(bp_high, nyq * 0.95)
    sos_bp = signal.butter(4, [bp_low / nyq, high_cut / nyq], btype="bandpass", output="sos")
    b_notch, a_notch = signal.iirnotch(notch_freq, Q=30.0, fs=fs_in)
    sos_lp = signal.butter(4, env_lp / nyq, btype="lowpass", output="sos")

    x = signal.sosfiltfilt(sos_bp, emg_raw, axis=0)
    x = signal.filtfilt(b_notch, a_notch, x, axis=0)
    x_env = signal.sosfiltfilt(sos_lp, np.abs(x), axis=0)
    x_env = np.clip(x_env, a_min=1e-8, a_max=None)  # strictly non-negative for NMF

    decim_factor = int(round(fs_in / fs_out))
    emg_env_ds = x_env[::decim_factor]
    pose_ds = pose_deg[::decim_factor]

    scale = np.percentile(emg_env_ds, 95) + 1e-8    # per-recording amplitude normalisation
    emg_env_norm = (emg_env_ds / scale).astype(np.float32)

    # FIX 2: inverse-kinematics failures are stored as frames where ALL 20 angles are 0
    # (same rule as emg2pose.utils.get_ik_failures_mask), not as NaN.
    pose_valid = ~np.all(np.isclose(pose_ds, 0.0), axis=1) & ~np.isnan(pose_ds).any(axis=1)
    return emg_env_norm, pose_ds.astype(np.float32), pose_valid

# Quick verification on the first recording
t_s, emg_sample, pose_sample, fs_est, stage = load_emg2pose_hdf5(h5_files[0], max_samples=40000)
qc = check_channel_artifacts(emg_sample)
emg_env, pose_ds, pose_valid = preprocess_emg_envelope(emg_sample, pose_sample, fs_in=fs_est)
print(f"Stage: {stage} | Sampling rate: {fs_est:.1f} Hz | Flagged channels: {qc['flagged_channels']}")
print(f"50 Hz envelope: {emg_env.shape} | pose: {pose_ds.shape} | valid pose frames: {pose_valid.mean():.1%}")

### Experiment 1: geometry of muscle synergies (linear NMF vs nonlinear autoencoder)

**Data kept per recording.** EMG-only models (NMF, AE, LSTM) use every frame; anything involving pose
uses only frames where pose tracking succeeded.

**Split (FIX 3).** Inside *each* recording: the first 80% is train, then a 1 s gap, then the last 20% is
test. This tests generalisation to **new time within the same stages and hands**, which is the easiest
condition and the right first check. The gap stops the 10 Hz smoothing from leaking across the boundary.
Cross-stage and cross-hand tests are separate, harder questions for `notebooks/02_how_many_synergies`.

In [ ]:
X_rec, Y_rec, V_rec, stage_rec = [], [], [], []
print("Processing HDF5 files...")
for f_path in h5_files:
    t_s, emg_raw, pose_deg, fs_est, stage = load_emg2pose_hdf5(f_path)
    emg_env, pose_ds, pose_valid = preprocess_emg_envelope(emg_raw, pose_deg, fs_in=fs_est)
    X_rec.append(emg_env); Y_rec.append(pose_ds); V_rec.append(pose_valid); stage_rec.append(stage)

GAP = 50  # frames = 1 s at 50 Hz
def split_blocks(arrs):
    train, test = [], []
    for a in arrs:
        cut = int(0.8 * len(a))
        train.append(a[:cut]); test.append(a[cut + GAP:])
    return train, test

X_tr_rec, X_te_rec = split_blocks(X_rec)
Y_tr_rec, Y_te_rec = split_blocks(Y_rec)
V_tr_rec, V_te_rec = split_blocks(V_rec)

X_train, X_test = np.concatenate(X_tr_rec), np.concatenate(X_te_rec)
Y_train, Y_test = np.concatenate(Y_tr_rec), np.concatenate(Y_te_rec)
V_train, V_test = np.concatenate(V_tr_rec), np.concatenate(V_te_rec)

print(f"{len(X_rec)} recordings, {len(set(stage_rec))} stages")
print(f"Train frames: {len(X_train)} ({V_train.mean():.1%} valid pose) | "
      f"Test frames: {len(X_test)} ({V_test.mean():.1%} valid pose)")

LATENT_DIM = 4  # number of synergies / latent units (sweep this in notebooks/02_how_many_synergies)

In [ ]:
# Linear synergies via NMF
print("Fitting NMF...")
nmf_model = NMF(n_components=LATENT_DIM, init="random", random_state=SEED, max_iter=500)
H_train_nmf = nmf_model.fit_transform(X_train)
H_test_nmf = nmf_model.transform(X_test)

X_recon_nmf = H_test_nmf @ nmf_model.components_
rec_r2_nmf = r2_score(X_test, X_recon_nmf, multioutput="variance_weighted")
print(f"NMF sEMG reconstruction R^2 (test): {rec_r2_nmf:.4f}")

In [ ]:
# Nonlinear synergies via a spatial autoencoder
class SpatialAutoencoder(nn.Module):
    def __init__(self, input_dim=16, latent_dim=4):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 32), nn.ReLU(),
            nn.Linear(32, 16), nn.ReLU(),
            nn.Linear(16, latent_dim), nn.ReLU(),   # non-negative latents, like NMF activations
        )
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 16), nn.ReLU(),
            nn.Linear(16, 32), nn.ReLU(),
            nn.Linear(32, input_dim), nn.ReLU(),    # non-negative reconstruction, like envelopes
        )

    def forward(self, x):
        latent = self.encoder(x)
        return self.decoder(latent), latent

torch.manual_seed(SEED)
train_loader = DataLoader(TensorDataset(torch.from_numpy(X_train)), batch_size=1024, shuffle=True)
ae_model = SpatialAutoencoder(input_dim=16, latent_dim=LATENT_DIM).to(DEVICE)
criterion = nn.MSELoss()
optimizer = optim.Adam(ae_model.parameters(), lr=1e-3, weight_decay=1e-5)

epochs = 40
ae_model.train()
print("Training autoencoder...")
for epoch in range(epochs):
    epoch_loss = torch.zeros((), device=DEVICE)
    for (x_batch,) in train_loader:
        x_batch = x_batch.to(DEVICE)
        optimizer.zero_grad()
        recon, _ = ae_model(x_batch)
        loss = criterion(recon, x_batch)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.detach() * x_batch.size(0)   # stays on the GPU: one sync per epoch
    if (epoch + 1) % 10 == 0 or epoch == 0:
        print(f"  Epoch {epoch+1:02d}/{epochs} | loss {epoch_loss.item() / len(X_train):.6f}")

In [ ]:
def ae_encode(X):
    ae_model.eval()
    with torch.no_grad():
        recon, latent = ae_model(torch.from_numpy(np.ascontiguousarray(X)).to(DEVICE))
    return recon.cpu().numpy(), latent.cpu().numpy()

X_recon_ae, H_test_ae = ae_encode(X_test)
_, H_train_ae = ae_encode(X_train)

rec_r2_ae = r2_score(X_test, X_recon_ae, multioutput="variance_weighted")
active = (H_test_ae.max(axis=0) > 1e-6).sum()
print(f"AE sEMG reconstruction R^2 (test): {rec_r2_ae:.4f}")
print(f"Active latent units: {active}/{LATENT_DIM}  (a unit that is always 0 is 'dead' and wasted)")

### Decoding hand pose from the latents

Every model gets the same Ridge decoder and is scored on the same valid test frames. Two reference
numbers (FIX 4) make the results interpretable:

- **Static baseline:** always predict the median training pose. Anything worse than this has learned nothing.
- **Full-envelope ridge:** Ridge on all 16 channels. This is the ceiling for a linear decoder; the
  4-D latents can only lose information relative to it.

R² measures point-by-point error; Pearson r measures whether the predicted trajectory has the right shape.

In [ ]:
def score(Y_true, Y_pred):
    r2 = r2_score(Y_true, Y_pred, multioutput="uniform_average")
    with np.errstate(invalid="ignore"):
        rs = [pearsonr(Y_true[:, j], Y_pred[:, j])[0] if np.ptp(Y_pred[:, j]) > 0 else np.nan
              for j in range(Y_true.shape[1])]
    return r2, np.nanmean(rs) if not np.all(np.isnan(rs)) else np.nan

Yv_train, Yv_test = Y_train[V_train], Y_test[V_test]
results = {}

# Static baseline: median training pose, every frame
static_pred = np.tile(np.median(Yv_train, axis=0), (len(Yv_test), 1))
results["Static (median pose)"] = score(Yv_test, static_pred)

# FIX 1: each decoder is trained AND tested on its own latents
for name, Ztr, Zte in [("Full 16-ch envelope", X_train, X_test),
                       ("NMF latents (k=4)", H_train_nmf, H_test_nmf),
                       ("AE latents (k=4)", H_train_ae, H_test_ae)]:
    dec = Ridge(alpha=1.0).fit(Ztr[V_train], Yv_train)
    results[name] = score(Yv_test, dec.predict(Zte[V_test]))

print(f"{'Representation':24s} {'pose R^2':>9s} {'Pearson r':>10s}")
for name, (r2, r) in results.items():
    print(f"{name:24s} {r2:9.4f} {r:10.4f}")

In [ ]:
names = list(results)
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].bar(["Linear NMF", "Nonlinear AE"], [rec_r2_nmf, rec_r2_ae], color=["#1f77b4", "#ff7f0e"])
axes[0].set_title("sEMG reconstruction ($R^2$, test)")

colors = ["#999999", "#555555", "#1f77b4", "#ff7f0e"]
axes[1].bar(names, [results[n][0] for n in names], color=colors)
axes[1].axhline(0, color="k", lw=0.8)
axes[1].set_title("Pose decoding ($R^2$)")

axes[2].bar(names, [results[n][1] for n in names], color=colors)
axes[2].set_title("Trajectory shape (Pearson $r$)")

for ax in axes[1:]:
    ax.tick_params(axis="x", rotation=20)
plt.tight_layout()
plt.show()

### Experiment 2: latent dynamics (next-step prediction)

Freeze the autoencoder, take its 4-D latents $z_t$, and train an LSTM to predict $z_{t+1}$ from the
previous 10 frames (200 ms).

- **FIX 5:** windows are built inside each recording's train or test block, so no window spans two recordings.
- **FIX 7:** latents are standardised with training statistics, so the loss is in comparable units.
- **FIX 4:** the **persistence baseline** predicts $z_{t+1} = z_t$. At 50 Hz on a 10 Hz-smoothed signal it is
  hard to beat, so the LSTM only "learns dynamics" if it beats this number.

In [ ]:
from numpy.lib.stride_tricks import sliding_window_view

SEQUENCE_LENGTH = 10

latent_scaler = StandardScaler().fit(H_train_ae)

def make_windows(blocks, L=SEQUENCE_LENGTH):
    # For each block: inputs z[i:i+L], target z[i+L]. Blocks are never joined.
    Xs, Ys = [], []
    for X_block in blocks:
        _, H = ae_encode(X_block)
        Z = latent_scaler.transform(H).astype(np.float32)
        if len(Z) <= L:
            continue
        Xs.append(sliding_window_view(Z[:-1], L, axis=0).transpose(0, 2, 1))  # (N, L, D)
        Ys.append(Z[L:])                                                         # (N, D)
    return np.concatenate(Xs).astype(np.float32), np.concatenate(Ys).astype(np.float32)

X_seq_train, Y_seq_train = make_windows(X_tr_rec)
X_seq_test, Y_seq_test = make_windows(X_te_rec)
print(f"Train windows: {X_seq_train.shape} | Test windows: {X_seq_test.shape}")

In [ ]:
class DynamicsLSTM(nn.Module):
    def __init__(self, latent_dim=4, hidden_dim=64):
        super().__init__()
        self.lstm = nn.LSTM(input_size=latent_dim, hidden_size=hidden_dim, num_layers=1, batch_first=True)
        self.fc = nn.Linear(hidden_dim, latent_dim)

    def forward(self, x):
        out, _ = self.lstm(x)
        return self.fc(out[:, -1, :])

torch.manual_seed(SEED)
train_seq_loader = DataLoader(TensorDataset(torch.from_numpy(X_seq_train), torch.from_numpy(Y_seq_train)),
                              batch_size=1024, shuffle=True)
rnn_model = DynamicsLSTM(latent_dim=LATENT_DIM, hidden_dim=64).to(DEVICE)
criterion_rnn = nn.MSELoss()
optimizer_rnn = optim.Adam(rnn_model.parameters(), lr=1e-3)

seq_epochs = 20
rnn_model.train()
print("Training dynamics model...")
for epoch in range(seq_epochs):
    epoch_loss = torch.zeros((), device=DEVICE)
    for x_b, y_b in train_seq_loader:
        x_b, y_b = x_b.to(DEVICE), y_b.to(DEVICE)
        optimizer_rnn.zero_grad()
        loss = criterion_rnn(rnn_model(x_b), y_b)
        loss.backward()
        optimizer_rnn.step()
        epoch_loss += loss.detach() * x_b.size(0)
    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f"  Epoch {epoch+1:02d}/{seq_epochs} | train MSE {epoch_loss.item() / len(X_seq_train):.6f}")

In [ ]:
rnn_model.eval()
with torch.no_grad():
    pred_test = rnn_model(torch.from_numpy(X_seq_test).to(DEVICE)).cpu().numpy()

mse_lstm = np.mean((pred_test - Y_seq_test) ** 2)
mse_persist = np.mean((X_seq_test[:, -1, :] - Y_seq_test) ** 2)   # z_{t+1} = z_t
mse_mean = np.mean(Y_seq_test ** 2)                                # always predict 0 (the training mean)
print(f"Test MSE (standardised units)")
print(f"  predict the mean:      {mse_mean:.4f}")
print(f"  persistence z(t+1)=z(t): {mse_persist:.4f}")
print(f"  LSTM:                  {mse_lstm:.4f}   ({1 - mse_lstm / mse_persist:+.1%} vs persistence)")

plt.figure(figsize=(12, 5))
for d, (c_true, c_pred) in enumerate([("#1f77b4", "#ff7f0e"), ("#2ca02c", "#d62728")]):
    plt.plot(Y_seq_test[:300, d], color=c_true, lw=2, label=f"Actual latent {d}")
    plt.plot(pred_test[:300, d], "--", color=c_pred, lw=2, label=f"Predicted latent {d}")
plt.title("One-step-ahead prediction on test data ($z_{t-9..t} \\to z_{t+1}$, teacher forcing)")
plt.xlabel("Time step (50 Hz)")
plt.ylabel("Latent activation (standardised)")
plt.legend()
plt.grid(True, linestyle="--", alpha=0.5)
plt.show()

### What this notebook can and cannot conclude

- **Can:** compare NMF and the AE fairly at k = 4 against a static baseline and a linear ceiling, and tell
  whether the LSTM beats "nothing changes".
- **Cannot yet:**
  - say the manifold is curved. That needs a k-sweep, PCA and a *linear* AE as controls, and 3+ seeds (`notebooks/02_how_many_synergies`).
  - say anything about self-sustaining vs input-driven dynamics. One-step teacher-forced prediction can't;
    that needs free rollouts and a model with EMG as input (`notebooks/06_synergy_dynamics`).